# Assignment Option C: Analyse an environmental sound
This notebook contains the executed code and explanations for Option C.

In [ ]:
# Single installation cell. Run before importing course libraries.
import sys, subprocess, os
assert (3,10) <= sys.version_info[:2] <= (3,12), 'This pinned environment targets Python 3.10–3.12. Use a compatible Colab runtime.'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_USE_LEGACY_KERAS'] = '1'
subprocess.check_call([sys.executable,'-m','pip','install','-q','torch==2.6.0','torchaudio==2.6.0','--index-url','https://download.pytorch.org/whl/cpu'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','torch==2.6.0','torchaudio==2.6.0','numpy==2.1.3','scipy==1.15.3','matplotlib==3.10.1','pandas==2.2.3','librosa==0.11.0','soundfile==0.13.1','scikit-learn==1.6.1','noisereduce==3.0.3','tensorflow==2.19.0','tf-keras==2.19.0','tensorflow-hub==0.16.1','openai-whisper==20250625','julius==0.2.7'])
print('Installation complete. If Colab requests a restart, restart the session, then continue below. Do not reinstall in the restarted session.')


In [ ]:
# Load the supplied ZIP once. No microphone is needed.
from pathlib import Path
import os, zipfile
ROOT = Path(os.environ.get('AUDIO_COURSE_ROOT','/content/audio_course'))
if not (ROOT/'data/manifest.csv').exists():
    from google.colab import files
    uploaded = files.upload()
    archives = [n for n in uploaded if n.lower().endswith('.zip')]
    assert len(archives)==1, 'Upload the supplied Audio_Course_Package.zip.'
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            destination=(ROOT/item.filename).resolve()
            assert destination.is_relative_to(ROOT.resolve()), 'Unexpected ZIP path.'
        z.extractall(ROOT)
assert (ROOT/'data/manifest.csv').exists(), 'Use the course package ZIP, which contains data/manifest.csv.'
print('Course folder:', ROOT)


In [ ]:
import os, json, time, re, unicodedata, importlib.metadata as meta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf
from scipy import signal
from IPython.display import Audio, display

os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_USE_LEGACY_KERAS']='1'
SEED=42
rng=np.random.default_rng(SEED)
FS=16000
DATA=ROOT/'data'
OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)

def load_audio(path, target_rate=FS):
    x, rate = sf.read(str(path),dtype='float32',always_2d=True)
    x=x.mean(axis=1)
    assert len(x)>0 and np.isfinite(x).all(), 'Invalid audio.'
    if rate != target_rate:
        x=librosa.resample(x,orig_sr=rate,target_sr=target_rate)
    return np.asarray(x,dtype=np.float32)

def play(x, rate=FS):
    display(Audio(np.asarray(x),rate=rate,normalize=False))

def save_plot(name):
    plt.tight_layout(); plt.savefig(FIG/name,dpi=140); plt.show()

def spectrum(x, fs=FS):
    a=np.asarray(x,dtype=float); a=a-a.mean()
    s=np.abs(np.fft.rfft(a*np.hanning(len(a))))
    return np.fft.rfftfreq(len(a),1/fs),20*np.log10(np.maximum(s/max(s.max(),1e-12),1e-6))

def inspect(x, fs=FS, title='Audio', name='inspection.png'):
    f,d=spectrum(x,fs)
    fig,ax=plt.subplots(2,1,figsize=(9,5))
    ax[0].plot(np.arange(len(x))/fs,x,lw=.7);ax[0].set(title=title,xlabel='Time (s)',ylabel='Amplitude')
    ax[1].plot(f,d);ax[1].set(xlabel='Frequency (Hz)',ylabel='Relative magnitude (dB)',xlim=(0,min(5000,fs/2)),ylim=(-80,3))
    save_plot(name)


## 1. Choose a recording, listen, and plot waveform/spectrum

In [ ]:
# Load manifest to find a dog recording (as requested by Option C)
manifest = pd.read_csv(DATA/'manifest.csv')
dog_row = manifest[manifest['label'] == 'dog'].iloc[0]
dog_file = dog_row['file']
print(f"Selected file: {dog_file}")

# Load the audio
audio_signal = load_audio(DATA/dog_file)

# Listen to the recording
play(audio_signal)

# Plot waveform and frequency spectrum
inspect(audio_signal, title=f'Waveform and Spectrum: {dog_file}', name='option_c_inspection.png')


## 2. Compare spectrograms (512 vs 2048 window lengths)

In [ ]:
# 16 kHz audio and 160-sample hop size for both
hop_size = 160

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
fig.subplots_adjust(hspace=0.4)

for ax, nfft in zip(axes, [512, 2048]):
    # Compute STFT
    S = np.abs(librosa.stft(audio_signal, n_fft=nfft, hop_length=hop_size, center=False))
    S = S / max(np.hanning(nfft).sum(), 1)
    db = librosa.amplitude_to_db(S, ref=1.0)
    
    # Plot spectrogram
    im = librosa.display.specshow(db, sr=FS, hop_length=hop_size, x_axis='time', y_axis='linear', ax=ax, vmin=-80, vmax=0)
    ax.set(title=f'Spectrogram (Window: {nfft} samples)', ylim=(0, FS/2))
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')

fig.colorbar(im, ax=axes, label='dB relative to amplitude 1')
save_plot('option_c_spectrograms.png')


## 3. Findings and Explanations

**Time Interval & Frequency Region:**
*Note: Run the cells above, then edit this section based on your specific dog bark plot.* 
- **Relevant time interval:** E.g., The primary bark occurs between ~0.5s and 1.2s.
- **Frequency region:** E.g., The highest concentration of energy lies roughly between 500 Hz and 2000 Hz.

**Effect of Window Length:**
The 512-sample window offers superior **time resolution**, meaning sharp transient sounds (like the onset of the bark) appear as crisp vertical lines without blurring across the time axis. However, its frequency resolution is lower. 
Conversely, the 2048-sample window provides better **frequency resolution**, revealing much clearer horizontal frequency bands and harmonics. The trade-off is that it smears these transient sounds across time, causing a loss of sharp temporal precision.

**Limitation of Analysis:**
This analysis investigates only a single recording. The temporal patterns, frequency concentrations, and harmonic structures observed here are highly specific to this particular dog and environment, and may not generalize accurately to other dog barks.

In [ ]:
# Zip the results folder for submission
import shutil
archive=shutil.make_archive(str(ROOT/'course_results'),'zip',OUT)
print('Results archive:',archive)
if 'google.colab' in __import__('sys').modules:
    from google.colab import files
    files.download(archive)
